# 01 — Data loading & cleaning

**Project question:** *Using only information available before kick-off, how well can we predict Premier League match outcomes (home win / draw / away win) — and how close can we get to the bookmakers?*

The bookmakers give us an unusually honest benchmark. Their odds encode probability estimates backed by real money, so instead of celebrating an accuracy number in a vacuum, we can ask: *how far are we from the practical ceiling?*

**Data source:** [football-data.co.uk](https://www.football-data.co.uk/englandm.php) publishes a free CSV per season with one row per match: the result, in-match statistics, and pre-match odds from several bookmakers. We use 11 complete seasons, **2015-16 through 2025-26** (11 × 380 = 4,180 matches).

**This notebook:** download the raw files, reduce 100+ columns to the ones we need, fix types, run sanity checks, and save one clean match table for the rest of the project.

In [1]:
import sys
sys.path.append("..")  # so we can import from src/

import pandas as pd

from src.data import SEASONS, RAW_DIR, download_raw, build_match_table, save_match_table

## 1. Download the raw data

Each season lives at a predictable URL (`.../mmz4281/<season>/E0.csv`, where `E0` is the code for the Premier League). The download function is **idempotent** — it skips files that already exist — so re-running this notebook never re-downloads or overwrites anything. The files in `data/raw/` are treated as read-only: every fix we make happens in code, downstream, so the whole pipeline can be reproduced from scratch.

In [2]:
download_raw()
sorted(p.name for p in RAW_DIR.glob("*.csv"))

['E0_1516.csv',
 'E0_1617.csv',
 'E0_1718.csv',
 'E0_1819.csv',
 'E0_1920.csv',
 'E0_2021.csv',
 'E0_2122.csv',
 'E0_2223.csv',
 'E0_2324.csv',
 'E0_2425.csv',
 'E0_2526.csv']

## 2. What does a raw file look like?

Always look at the raw data before touching it. One season file has 380 rows (matches) and over 100 columns, most of which are odds from different bookmakers.

In [3]:
raw = pd.read_csv(RAW_DIR / "E0_2526.csv")
print(f"shape: {raw.shape}")
print(f"first 25 columns: {list(raw.columns[:25])}")
raw.head(3)

shape: (380, 132)
first 25 columns: ['Div', 'Date', 'Time', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'FTR', 'HTHG', 'HTAG', 'HTR', 'Referee', 'HS', 'AS', 'HST', 'AST', 'HF', 'AF', 'HC', 'AC', 'HY', 'AY', 'HR', 'AR', 'B365H']


,Div,Date,Time,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,...,B365CAHH,B365CAHA,PCAHH,PCAHA,MaxCAHH,MaxCAHA,AvgCAHH,AvgCAHA,BFECAHH,BFECAHA
0,E0,15/08/2025,20:00,Liverpool,Bournemouth,4,2,H,1,0,...,2.03,1.78,2.07,1.85,2.03,1.88,1.94,1.76,2.14,1.86
1,E0,16/08/2025,12:30,Aston Villa,Newcastle,0,0,D,0,0,...,2.05,1.80,2.02,1.89,2.06,1.80,1.95,1.74,2.14,1.86
2,E0,16/08/2025,15:00,Brighton,Fulham,1,1,D,0,0,...,1.83,2.03,1.93,2.00,1.84,2.03,1.80,1.96,1.91,2.08


### Which columns do we keep, and why?

The column names are terse codes ([documented here](https://www.football-data.co.uk/notes.txt)). We keep three groups:

| Group | Raw columns | Why we keep them |
|---|---|---|
| **Identity** | `Date`, `HomeTeam`, `AwayTeam` | Who played, when — needed to build team-form features later |
| **Outcome & match stats** | `FTHG`, `FTAG`, `FTR`, `HS`, `AS`, `HST`, `AST`, `HC`, `AC` | Full-time goals/result, shots, shots on target, corners |
| **Bookmaker odds** | `B365H`, `B365D`, `B365A` | Bet365 pre-match odds — our benchmark to beat |

We use one bookmaker (Bet365) rather than averaging across all of them: it's present in every season, and one consistent benchmark keeps the comparison clean.

⚠️ **A note on data leakage.** The match statistics (shots, corners…) describe what happened *during* the match — they don't exist at prediction time. They are kept for exploratory analysis and for building *historical* form features (e.g., "shots on target over the last 5 matches"), but the raw per-match values must **never** be direct model inputs. Getting this wrong is the classic way to build a football model that looks brilliant and is actually useless.

## 3. Build one clean match table

`build_match_table()` (in [`src/data.py`](../src/data.py)) does the actual work, so the exact same cleaning code can be reused by the Streamlit app later. It:

1. reads all 11 season files and keeps/renames the columns above;
2. adds a `season` label to each row;
3. parses dates — a real-world wrinkle: older seasons use 2-digit years (`13/08/15`), newer ones 4-digit (`13/08/2015`), and both are day-first. `pd.to_datetime(..., dayfirst=True, format="mixed")` handles both;
4. sorts chronologically — this matters because all our features and our train/test split will be time-based.

In [4]:
matches = build_match_table()
print(f"{len(matches)} matches, {matches['date'].min().date()} to {matches['date'].max().date()}")
matches.head()

4180 matches, 2015-08-08 to 2026-05-24


,date,home_team,away_team,home_goals,away_goals,result,home_shots,away_shots,home_shots_on_target,away_shots_on_target,home_corners,away_corners,odds_home,odds_draw,odds_away,season
0,2015-08-08,Bournemouth,Aston Villa,0,1,A,11,7,2,3,6,3,2.00,3.6,4.00,2015-16
1,2015-08-08,Chelsea,Swansea,2,2,D,11,18,3,10,4,8,1.36,5.0,11.00,2015-16
2,2015-08-08,Everton,Watford,2,2,D,10,11,5,5,8,2,1.70,3.9,5.50,2015-16
3,2015-08-08,Leicester,Sunderland,4,2,H,19,10,8,5,6,3,1.95,3.5,4.33,2015-16
4,2015-08-08,Man United,Tottenham,1,0,H,9,9,1,4,1,2,1.65,4.0,6.00,2015-16


## 4. Sanity checks

Cleaning isn't done until we've *verified* the data matches reality. Things we know must be true about the Premier League:

- every season has exactly **380 matches** (20 teams, each plays the other 19 home and away);
- every season has exactly **20 distinct teams**;
- the `result` column must agree with the goals (`H` ⇔ home_goals > away_goals, etc.);
- no missing values, no duplicated matches;
- odds must all be > 1.0 (an odd of exactly 1.0 would mean a guaranteed outcome).

In [5]:
# 380 matches and 20 teams per season
per_season = matches.groupby("season").agg(
    n_matches=("result", "size"),
    n_teams=("home_team", "nunique"),
)
assert (per_season["n_matches"] == 380).all()
assert (per_season["n_teams"] == 20).all()
per_season

,n_matches,n_teams
season,,
2015-16,380,20
2016-17,380,20
2017-18,380,20
2018-19,380,20
2019-20,380,20
2020-21,380,20
2021-22,380,20
2022-23,380,20
2023-24,380,20


In [6]:
# result agrees with the goals in every single row
expected = pd.Series("D", index=matches.index)
expected[matches["home_goals"] > matches["away_goals"]] = "H"
expected[matches["home_goals"] < matches["away_goals"]] = "A"
assert (matches["result"] == expected).all()

# completeness, uniqueness, plausible odds
assert matches.isna().sum().sum() == 0
assert not matches.duplicated(subset=["date", "home_team", "away_team"]).any()
assert (matches[["odds_home", "odds_draw", "odds_away"]] > 1.0).all().all()

print("all checks passed ✔")
print(f"missing values: {matches.isna().sum().sum()}")
print(f"result distribution:\n{matches['result'].value_counts(normalize=True).round(3).to_string()}")

all checks passed ✔
missing values: 0
result distribution:
result
H    0.443
A    0.320
D    0.237


The result distribution is our first real finding: **home wins ~44%, away wins ~33%, draws ~23%**. Two things follow immediately:

1. a lazy model that always predicts "home win" is already ~44% accurate — that's the baseline any real model must beat;
2. the three classes are imbalanced, so overall accuracy alone will be a misleading metric. We'll need per-class precision/recall (spoiler: draws are notoriously hard).

## 5. Save the clean table

In [7]:
out_path = save_match_table(matches)
print(f"saved {len(matches)} matches to {out_path}")

saved 4180 matches to /Users/abdullahboraei/Desktop/DataScience/epl-match-prediction/data/processed/matches.csv


**Next:** [02_eda.ipynb](02_eda.ipynb) — how big is home advantage, how often do favourites actually win, and what happened to home advantage in the empty-stadium COVID season?